In [ ]:
import re
import os
import warnings
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

# NLTK imports
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet, stopwords
from nltk import pos_tag
from spacy.lang.en.stop_words import STOP_WORDS as SPACY_STOP_WORDS

# Locate the project root and define paths consistently with the other pipelines.
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)

    for p in [start.resolve()] + list(start.resolve().parents):
        if (p / "csvs").exists() and (p / "markdown").exists():
            return p

    raise FileNotFoundError("Project root not found. Run inside the MSC project.")


PROJECT_ROOT = find_project_root()

# Make the project root importable.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts.text_mining_utils as tmu

CSVS_CHUNKED = PROJECT_ROOT / "csvs" / "chunked"
CHUNKS_ALL = CSVS_CHUNKED / "chunks_all.csv"

PIPELINE_DIR = PROJECT_ROOT / "progress" / "exploration" / "policy"
OUT_DIR = PIPELINE_DIR / "img"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
warnings.filterwarnings('ignore')

NLTK_DATA_DIR = PROJECT_ROOT / "data" / "auto"
NLTK_DATA_DIR.mkdir(parents=True, exist_ok=True)
nltk.data.path.append(str(NLTK_DATA_DIR))

# Download NLTK data
nltk_data = [
    'wordnet', 'omw-1.4', 'punkt', 'punkt_tab',
    'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'stopwords'
]
for package in nltk_data:
    try:
        nltk.download(package, download_dir=str(NLTK_DATA_DIR), quiet=True)
    except Exception as e:
        print(f"Warning: Could not download {package}: {e}")

print("Project root:", PROJECT_ROOT)
print("Input chunks file:", CHUNKS_ALL)
print("Pipeline folder:", PIPELINE_DIR)
print("Output folder:", OUT_DIR)
print("Utility module:", tmu.__name__)


In [ ]:
# ==========================================
# Loading data
# ==========================================
try:
    print("Project root:", PROJECT_ROOT)
    print("Expected chunks file:", CHUNKS_ALL)
    print("File exists:", CHUNKS_ALL.exists())

    df = pd.read_csv(CHUNKS_ALL, encoding="utf-8-sig")

    UNESCO_DOC_IDS = {
        "policy_ai_competency_framework_for_students",
        "policy_ai_competency_framework_for_teachers",
    }

    unesco_mask = df["doc_id"].isin(UNESCO_DOC_IDS)
    df.loc[unesco_mask, "country"] = "unesco"

    df = df.rename(columns={
        "heading_context": "Title",
        "chunk_text": "Content"
    })

    df["Title"] = df["Title"].fillna("").astype(str)

    df = df[df["Content"] != "N/A"]
    df = df.dropna(subset=["Content"])

    df["Content"] = df["Title"] + " " + df["Content"]

    print(f"Successfully loaded {len(df)} documents.")

except FileNotFoundError:
    print("Error: chunks_all.csv not found.")
    print("Expected path:", CHUNKS_ALL)
    raise

except Exception as e:
    print(f"An error occurred: {e}")
    raise

In [ ]:
# ==========================================
# Filter for Policy Data Only
# ==========================================

# Filter based on the 'corpus' column
df = df[df['corpus'] == 'policy']

print("-" * 30)
# ==========================================

# Ensure titles are strings and handle missing values
df['Title'] = df['Title'].fillna('').astype(str)

# Remove rows with failed extraction or missing content
df = df[df['Content'] != 'N/A']
df = df.dropna(subset=['Content'])

# Combine Title and Content for analysis
df['Content'] = df['Title'] + " " + df['Content']

print(f"Successfully loaded {len(df)} POLICY documents.")


In [ ]:
# ==========================================
# 3. Define Custom Stopwords 
# ==========================================

# Base stop words initialization
stop_words = set(SPACY_STOP_WORDS)
stop_words.update(set(stopwords.words('english')))

# Add French Stopwords
try:
    french_stopwords = set(stopwords.words('french'))
    stop_words.update(french_stopwords)
except:
    print("Warning: French stopwords not found.")

# ============================================================
# Reorganized custom stopwords
# Goal:
# - Keep the same stopword vocabulary
# - Make categories clearer and more accurate
# - Apply these AFTER clean_text()
# ============================================================


# ------------------------------------------------------------
# 1. PDF, scraping, formatting, and document extraction artifacts
# ------------------------------------------------------------
document_artifact_stopwords = [
    'pdf', 'qxp', 'page', 'copyright', 'reserved',
    'www', 'http', 'https', 'com', 'org',
    'figure', 'table', 'annex', 'appendix', 'footnote', 'endnote',
    'isbn', 'bibliography', 'citation', 'reference', 'metadata', 'docx',
    'executive summary', 'draft', 'matrix', 'blueprint', 'snapshot',
    'chart', 'brochure',
    'fig'
]


# ------------------------------------------------------------
# 2. Legal and policy document structure words
# These usually describe the document format, not the policy topic
# ------------------------------------------------------------
legal_structure_stopwords = [
    'section', 'chapter', 'part', 'title', 'article', 'clause',
    'paragraph', 'subparagraph', 'schedule', 'exhibit', 'supplement',
    'addendum', 'preamble', 'recital', 'report',

    # French structure words
    'chapitre', 'partie', 'titre', 'paragraphe', 'alinea',
    'annexe', 'appendice', 'expos', 'des', 'motifs',
    'disposition'
]


# ------------------------------------------------------------
# 3. Legal / bureaucratic boilerplate
# These are legal phrasing words, not substantive keywords
# ------------------------------------------------------------
legal_boilerplate_stopwords = [
    'coppa', 'ferpa', 'enact', 'bit',
    'herein', 'therein', 'thereof', 'whereof', 'hereby',
    'whereby', 'hereinafter', 'thereafter', 'aforementioned',
    'foregoing', 'henceforth', 'whereto', 'whereupon',
    'notwithstanding', 'pursuant', 'thereto', 'thereunder', 'hereto',

    # French legal / OCR-normalized forms
    'susmentionn', 'prcit', 'nonobstant', 'par drogation',
    'en', 'consquence', 'eu', 'gard', 'aux fins',
    'selon', 'conformment', 'outre', 'ladite', 'ledit', 'dune'
]


# ------------------------------------------------------------
# 4. Modal, obligation, and policy-action verbs
# These often dominate policy texts but are not domain keywords
# ------------------------------------------------------------
policy_action_stopwords = [
    'shall', 'must', 'may', 'might', 'could', 'would', 'should', 'will',
    'require', 'requires', 'required',
    'ensure', 'ensures',
    'seek', 'seeks', 'seeking',
    'establish', 'establishes',
    'promote', 'promotes', 'promoting',
    'maintain', 'respect',

    # French equivalents / OCR-normalized forms
    'doit', 'devrait', 'pourrait', 'vouloir',
    'exiger', 'requis', 'assurer', 'garantir',
    'rechercher', 'tablir', 'promouvoir',
    'maintenir', 'respecter'
]


# ------------------------------------------------------------
# 5. Generic policy connectors and vague qualifiers
# These help grammar but rarely reveal the policy theme
# ------------------------------------------------------------
policy_connector_stopwords = [
    'various', 'several', 'multiple', 'certain', 'relevant', 'appropriate',
    'regarding', 'concerning', 'involving', 'accordance',
    'relate', 'relates', 'relating',
    'furthermore', 'moreover', 'therefore', 'thus', 'hence',
    'however', 'although', 'including', 'etc', 'etcetera',

    # French equivalents
    'concernant', 'divers', 'plusieurs', 'particulier',
    'certain', 'aussi', 'de plus', 'en outre',
    'par consquent', 'donc', 'ainsi', 'cependant',
    'bien que', 'notamment', 'entre autres', 'comme',
    'toute', 'tous', 'cette', 'plus', 'leurs'
]


# ------------------------------------------------------------
# 6. Geographic, national, governmental, and institutional noise
# Useful to remove if you compare policy themes rather than countries
# ------------------------------------------------------------
geo_institutional_stopwords = [
    'ireland', 'irish', 'france', 'french', 'usa', 'america',
    'australia', 'australian',
    'country', 'nation', 'national', 'state',
    'government', 'federal', 'ministry', 'department',
    'gouvernement', 'ministre', 'dtat', 'republic',
    'pittsburg', 'antioch', 'medina', 'winterset',
    'sara', 'washington', 'california', 'massachusetts',
    'oregon', 'sweden', 'unified', 'district',
    'commonwealth', 'nsw'
]


# ------------------------------------------------------------
# 7. AI, digital, technology, vendor, and model-name noise
# Be careful: remove these only if you want deeper policy themes,
# not surface AI vocabulary.
# ------------------------------------------------------------
tech_ai_stopwords = [
    'artificial', 'intelligence',
    'ai', 'ia', 'lia', 'lIA', 'dia', 'dIA',
    'genai', 'generative',
    'model', 'models', 'llm',
    'algorithm', 'algorithms',
    'machine', 'learning',
    'data', 'dataset', 'datasets',
    'digital', 'numrique',
    'computer', 'software', 'hardware',
    'technology', 'technological',
    'platform', 'online', 'internet', 'web', 'electronic',
    'tool', 'system', 'device', 'application', 'app', 'service',
    'broadband', 'connectivity', 'vendor', 'player', 'enterprise',

    # French / OCR-normalized technical terms
    'artificielle', 'gen', 'generatif',
    'modle', 'modles',
    'apprentissage', 'donn',
    'ordinateur', 'logiciel', 'matriel',
    'technologie', 'plateforme', 'ligne',
    'ectronique', 'outil', 'systme', 'dispositif',
    'générative', 'intelligent',

    # Vendors, systems, and model-related terms
    'edge', 'gpt', 'openai', 'microsoft', 'google',
    'amazon', 'meta', 'ibm', 'nvidia', 'intel',
    'gemini', 'deepseek', 'anthropic', 'claude',
    'embedding', 'transformer', 'natural language',
    'climate change', 'ieee', 'architecture',
    'chatgtp', 'copilot', 'kwyk', 'technologies',

    # Extra English terms seen in the French corpus
    'new', 'work', 'law', 'system', 'systems', 'right',
    'services', 'based', 'case', 'support', 'human',
    'generated', 'related'
]


# ------------------------------------------------------------
# 8. Education-system generic terms
# Remove these if the whole corpus is already about education.
# Keep them if you want education-specific keywords.
# ------------------------------------------------------------
education_stopwords = [
    'school', 'education', 'educational',
    'pupil', 'classroom', 'learning',
    'teacher', 'teachers', 'schools',
    'student', 'students',
    'post', 'primary', 'technical',

    # French education terms and OCR variants
    'cole', 'educatif', 'enseignant', 'enseignats', 'enseignants',
    'professeur', 'programme',
    'apprentissage', 'enseignement', 'formation',
    'établissement', 'éducation', 'école',
    'élève', 'élèves',

    # Other education/project terms
    'covid', 'showcase', 'deliverable'
]


# ------------------------------------------------------------
# 9. Organizations, people, acronyms, and named entities
# ------------------------------------------------------------
org_name_stopwords = [
    'adam', 'mary', 'jensen', 'abraham', 'lincoln',
    'knight', 'kim', 'guez',
    'oecd', 'cbs', 'webwise', 'cosn', 'dlf',
    'scoilnet', 'deap', 'dne', 'men',
    'parliament', 'commissioner', 'public sector',
    'cipa', 'protection act', 'oide', 'tie', 'cpd'
]


# ------------------------------------------------------------
# 10. Generic vague words and corpus-specific noise
# ------------------------------------------------------------
generic_noise_stopwords = [
    'feel', 'like', 'everybody', 'actually',
    'kid', 'employee', 'min', 'box', 'loop',
    'black', 'asset', 'typical', 'era',
    'acquire', 'behavioral', 'behavioural',
    'revolution', 'emission', 'talent',
    'billion', 'count', 'witness', 'operator',
    'interim', 'resourced', 'resourcing',
    'round', 'support work', 'dashboard',
    'ela', 'estonia', 'pisa', 'perspectives',
    'final', 'total', 'describe', 'visit',
    'original', 'sample', 'low',
    'answer', 'ask', 'efficacy',
    'deepen', 'enhance',
    'author', 'researcher', 'mission', 'personnel',
    'palo', 'alto', 'card',
    'use', 'lot', 'enable', 'constituent',
    'session', 'policy', 'document',
    'member', 'organisation', 'goal',
    'long term', 'desirable', 'young',
    'ite', 'aied', 'australasian',
    'perplexed', 'guidetoaiinschools', 'nenufsd',
    'dent', 'fee', 'want', 'prof', 'stu'
]


# ------------------------------------------------------------
# 11. Dates and temporal noise
# ------------------------------------------------------------
date_time_stopwords = [
    'july', 'april', 'august'
]


# ------------------------------------------------------------
# 12. French and English generic content words seen after cleaning
# These should be applied AFTER clean_text()
# ------------------------------------------------------------
post_cleaning_generic_stopwords = [
    'être', 'avoir', 'faire', 'peut',
    'tout', 'tous', 'toute', 'toutes',
    'autre', 'autres',
    'encore', 'déjà', 'non', 'sans',
    'dont', 'cela', 'même', 'mme',
    'très', 'moins', 'plus', 'souvent',
    'cas', 'type', 'manière', 'fonction', 'partir',
    'exemple', 'question', 'réponse', 'résultat',
    'objectif', 'contenu', 'texte', 'image'
]


# ------------------------------------------------------------
# 13. Word fragments and OCR leftovers
# ------------------------------------------------------------
fragment_stopwords = [
    'ing', 'tion', 'ment', 'ness', 'ly',
    'able', 'ive', 'ent', 'ant',
    'ence', 'ance',
    'ali', 'gue', 'rod', 'rodr',
    'educa', 'evi', 'al'
]


# ------------------------------------------------------------
# Merge all custom stopword categories
# ------------------------------------------------------------
custom_stopword_categories = [
    document_artifact_stopwords,
    legal_structure_stopwords,
    legal_boilerplate_stopwords,
    policy_action_stopwords,
    policy_connector_stopwords,
    geo_institutional_stopwords,
    tech_ai_stopwords,
    education_stopwords,
    org_name_stopwords,
    generic_noise_stopwords,
    date_time_stopwords,
    post_cleaning_generic_stopwords,
    fragment_stopwords,
]

custom_stopwords = set()

for category in custom_stopword_categories:
    custom_stopwords.update(category)


# Add to your existing stop_words
stop_words.update(custom_stopwords)

# ==========================================
# 4. Text Preprocessing
# ==========================================

# Initialize Lemmatizer
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def preprocess_text(text):
    # Lowercase and remove non-alphabetic characters
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    
    # Tokenize
    tokens = nltk.word_tokenize(text)
    
    # Remove stopwords and short words
    tokens = [token for token in tokens if token not in stop_words and len(token) > 2]
    
    # POS Tagging
    pos_tags = pos_tag(tokens)
    
    # Lemmatize
    clean_tokens = [lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in pos_tags]
    
    return " ".join(clean_tokens)

# Do NOT create Processed_Content here.
# The keyword pipeline below must call clean_text() first, then remove stopwords.
print("Stopwords loaded. Processed_Content will be built after clean_text() in the next cell.")
print(f"Shape before keyword cleaning: {df.shape}")


In [ ]:

# ==========================================
# Basic cleaning function + keyword-ready text
# ==========================================

def clean_text(text):
    """
    Normalise policy text before keyword extraction.

    Important design choice:
    - This function fixes OCR/extraction issues and French contractions first.
    - Stop words are NOT changed here.
    - Stop words are removed only after clean_text() has produced normalised tokens.

    This order is important for the French corpus because forms such as l’IA, l'IA,
    d’IA and d'IA first become normal tokens such as ia, then the existing
    stop_words list can remove ia / lia / dia-style artefacts consistently.
    """
    # Lowercase first to match stopwords later
    text = str(text).lower()

    # Normalise apostrophes so l’ia and l'ia are treated the same way
    text = text.replace("’", "'").replace("‘", "'").replace("`", "'")

    # Remove common extraction artefacts
    text = text.replace("guillemetleft", " ")
    text = text.replace("guillemetright", " ")

    # Fix frequent OCR / extraction artefacts observed in French outputs
    ocr_fixes = {
        r"\blve\b": "élève",
        r"\blves\b": "élèves",
        r"\bducation\b": "éducation",
        r"\b1ducation\b": "éducation",
        r"\bducatif\b": "éducatif",
        r"\bducative\b": "éducative",
        r"\bducatifs\b": "éducatifs",
        r"\bducatives\b": "éducatives",
        r"\btablissement\b": "établissement",
        r"\btablissements\b": "établissements",
        r"\bvaluation\b": "évaluation",
        r"\bvaluer\b": "évaluer",
        r"\btrs\b": "très",
        r"\blment\b": "élément",
        r"\blments\b": "éléments",
    }

    for pattern, replacement in ocr_fixes.items():
        text = re.sub(pattern, replacement, text)

    # Expand common French contractions before removing punctuation.
    # Example: l'IA -> la IA -> tokens: la, ia.
    french_contractions = {
        r"\bl'": "la ",
        r"\bd'": "de ",
        r"\bc'": "ce ",
        r"\bj'": "je ",
        r"\bm'": "me ",
        r"\bt'": "te ",
        r"\bs'": "se ",
        r"\bn'": "ne ",
        r"\bqu'": "que ",
        r"\blorsqu'": "lorsque ",
        r"\bpuisqu'": "puisque ",
        r"\bquoiqu'": "quoique ",
        r"\bjusqu'": "jusque ",
        r"\baujourd'": "aujourd ",
        r"\bpresqu'": "presque ",
    }

    for pattern, replacement in french_contractions.items():
        text = re.sub(pattern, replacement, text)

    # Remove email addresses
    text = re.sub(r"\S*@\S*\s?", " ", text)

    # Remove URLs and HTML tags
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text, flags=re.MULTILINE)
    text = re.sub(r"<.*?>", " ", text)

    # Remove reference citations
    text = re.sub(r"\[\d+\]", " ", text)
    text = re.sub(r"\(\s*see\s+[^\)]+\)", " ", text)
    text = re.sub(r"footnote\s*\d+", " ", text)

    # Remove standalone numbers
    text = re.sub(r"\b\d+\b", " ", text)

    # Remove HTML entities
    text = re.sub(r"\b(amp|lt|gt|quot|nbsp)\b", " ", text)

    # Remove punctuation/separators while keeping French accented letters
    text = re.sub(r"[^a-zàâäéèêëïîôùûüÿçæœ\s-]", " ", text)

    # Return normalised tokens only. Stopword removal happens after this call.
    return text.split()


# Important short policy acronyms to preserve only if they are not stopwords.
# Because 'ai' and 'ia' are already in stop_words, they will still be removed.
important_short_terms = {"ai", "ia", "eu", "uk", "us", "un", "ocde", "oecd"}

# Extra French function words created by contraction expansion.
# This does not change stop_words; it is only a post-cleaning filter.
extra_post_clean_stopwords = {
    "la", "le", "les", "de", "un", "une", "des", "du", "au", "aux",
    "ce", "je", "me", "te", "se", "ne", "que"
}


def remove_stopwords_after_clean_text(tokens):
    """
    Remove the existing stop_words AFTER clean_text() normalisation.
    This is the step that removes French artefacts such as dia/lia when they
    appear after apostrophe/contraction normalisation.
    """
    filtered_tokens = []

    for token in tokens:
        word = token.strip("-")

        if not word:
            continue

        # Keep meaningful short acronyms only when they are not stop words.
        long_enough = len(word) > 2 or word in important_short_terms
        if not long_enough:
            continue

        # Do not change stop_words: just apply the existing list here.
        if word in stop_words:
            continue

        if word in extra_post_clean_stopwords:
            continue

        filtered_tokens.append(word)

    return filtered_tokens


def build_keyword_text(text):
    """Full keyword-preparation pipeline: clean first, then remove stopwords."""
    tokens = clean_text(text)
    keyword_tokens = remove_stopwords_after_clean_text(tokens)
    return " ".join(keyword_tokens)


print(f"Cleaning {len(df)} documents with clean_text() first, then removing stopwords...")

# Apply the keyword-preparation pipeline.
df['Clean_Tokens'] = df['Content'].apply(clean_text)
df['Processed_Content'] = df['Clean_Tokens'].apply(lambda tokens: " ".join(remove_stopwords_after_clean_text(tokens)))

# Keep this alias for compatibility with earlier notebook cells.
df['Clean_Content'] = df['Processed_Content']

# Filter out empty documents after keyword preparation.
df = df[df['Processed_Content'].str.len() > 0].copy()

print(f"Valid documents after cleaning: {len(df)}")
print("Check artefacts removed after clean_text():")
for artefact in ["dia", "lia", "ia", "ai"]:
    count = sum(text.split().count(artefact) for text in df['Processed_Content'])
    print(f"  {artefact}: {count}")


In [ ]:
# ==========================================
# Basic statistics
# ==========================================

# Calculate word count based on the processed text column
df['Word_Count'] = df['Processed_Content'].apply(lambda x: len(x.split()))

total_words = df['Word_Count'].sum()
avg_words_per_article = df['Word_Count'].mean()

# 'country' is lowercase in your original CSV structure
unique_countries = df['country'].nunique()

print("Basic statistics")
print(f"Total documents/rows: {len(df)}")
print(f"Total unique countries: {unique_countries}")
print(f"Total words (after cleaning): {total_words}")
print(f"Average words per document: {avg_words_per_article:.2f}")

In [ ]:
# ==========================================
# Exploratory data analysis
# ==========================================

# Create the directory if it doesn't exist
output_dir = str(OUT_DIR)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

# Set general style for plots
sns.set(style="whitegrid")

# A. Distribution of Document Lengths
plt.figure(figsize=(10, 6))
sns.histplot(df['Word_Count'], bins=30, kde=True, color='skyblue')
plt.xlabel('Number of words')
plt.ylabel('Frequency')

# Save the image
save_path = os.path.join(output_dir, 'word_count_distribution.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"Saved: {save_path}")

plt.show()
plt.close()

# B. Document Count per Country
plt.figure(figsize=(12, 6))

# Limit to top 10 countries to keep the chart readable
# FIX: Changed 'Country' to 'country' (lowercase) to match CSV column names
top_countries = df['country'].value_counts().nlargest(10).index

sns.countplot(
    data=df[df['country'].isin(top_countries)], 
    x='country',  # FIX: Changed 'Country' to 'country'
    order=top_countries, 
    palette='viridis'
)

plt.xlabel('Country')
plt.ylabel('Number of chunks')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Save the image
save_path = os.path.join(output_dir, 'doc_count_per_country.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"Saved: {save_path}")

plt.show()
plt.close()

In [ ]:

# ==========================================
# Global word cloud: keyword-ready policy text
# ==========================================

output_dir = str(OUT_DIR)
os.makedirs(output_dir, exist_ok=True)

def plot_wordcloud(text_data, title, max_words=100, filename=None):
    """Generates, saves, and plots a word cloud from already-cleaned keyword text."""
    if not text_data.strip():
        print("No keyword text available for word cloud.")
        return

    wordcloud = WordCloud(
        background_color='white',
        width=800,
        height=400,
        max_words=max_words
    ).generate(text_data)
    
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title)
    
    if filename:
        save_path = os.path.join(output_dir, filename)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()
    plt.close()

# Use Processed_Content, not raw Content, so the word cloud reveals real policy keywords.
all_text_combined = " ".join(df['Processed_Content'])

plot_wordcloud(
    all_text_combined,
    '',
    filename='global_wordcloud.png'
)


In [ ]:

# ==========================================
# Word cloud per country: keyword-ready policy text
# ==========================================

output_dir = str(OUT_DIR)
os.makedirs(output_dir, exist_ok=True)

top_countries = df['country'].value_counts().nlargest(4).index

def plot_wordcloud(text, title, filename):
    if not text.strip():
        print(f"Skipping {title}: no keyword text available.")
        return

    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap='viridis'
    ).generate(text)
    
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.tight_layout(pad=0)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {filename}")

for country in top_countries:
    country_text = " ".join(df[df['country'] == country]['Processed_Content'])
    
    if len(country_text.strip()) == 0:
        print(f"Skipping {country}: No processed keyword text available.")
        continue

    filename = os.path.join(output_dir, f'wordcloud_{str(country).lower()}.png')
    
    plot_wordcloud(
        country_text,
        f'Top Words in Country: {country}',
        filename=filename
    )


In [ ]:

# ==========================================
# Top 20 unigrams bar chart: real policy keywords
# ==========================================

output_dir = str(OUT_DIR)
os.makedirs(output_dir, exist_ok=True)

all_text_combined = " ".join(df['Processed_Content'])
all_words_list = all_text_combined.split()

# Processed_Content has already been cleaned and stopword-filtered after clean_text().
word_counts = Counter(all_words_list)
most_common_words = word_counts.most_common(20)

if len(most_common_words) > 0:
    words, counts = zip(*most_common_words)
else:
    words, counts = [], []
    print("No words found to display.")

print("\nText Output Preview: Top 20 policy keywords")
for word, count in most_common_words:
    print(f"{word}: {count}")

if len(words) > 0:
    plt.figure(figsize=(20, 6))
    sns.barplot(x=list(words), y=list(counts), palette='rocket')
    plt.title('Top 20 Policy Keywords', fontsize=15)
    plt.ylabel('Frequency')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

    save_path = os.path.join(output_dir, 'top_20_policy_keywords.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {save_path}")

    plt.show()
    plt.close()


In [ ]:

# ==========================================
# Top 20 policy keywords per country
# ==========================================

output_dir = str(OUT_DIR)
os.makedirs(output_dir, exist_ok=True)

countries = df['country'].dropna().unique()

for country in countries:
    country_df = df[df['country'] == country]
    
    if country_df.empty:
        print(f"Skipping {country}: No data found.")
        continue

    text_content = " ".join(country_df['Processed_Content'])
    
    if not text_content.strip():
        print(f"Skipping {country}: No keyword text content.")
        continue

    words_list = text_content.split()
    word_counts = Counter(words_list)
    most_common_words = word_counts.most_common(20)
    
    if not most_common_words:
        print(f"Skipping {country}: No words found after processing.")
        continue
    
    print(f"\nTop 20 Policy Keywords in {country}")
    for word, count in most_common_words:
        print(f"{word}: {count}")
    
    words, counts = zip(*most_common_words)
    
    plt.figure(figsize=(20, 6))
    sns.barplot(x=list(words), y=list(counts), palette='rocket')
    plt.title(f'Top Policy Keywords: {country}')
    plt.xlabel('Keywords')
    plt.ylabel('Frequency')
    plt.xticks(rotation=45, ha='right') 
    plt.tight_layout()
    
    safe_country_name = str(country).lower().replace(" ", "_").replace("/", "_")
    filename = f'top_policy_keywords_{safe_country_name}.png'
    save_path = os.path.join(output_dir, filename)
    
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {save_path}")
    
    plt.show()
    plt.close()


In [ ]:

# ==========================================
# Optional: Top policy keyword phrases (bigrams/trigrams)
# ==========================================

from collections import Counter


def ngram_counts(texts, n=2, top_k=30):
    counts = Counter()
    for text in texts:
        tokens = text.split()
        counts.update(zip(*[tokens[i:] for i in range(n)]))
    return [(" ".join(gram), count) for gram, count in counts.most_common(top_k)]

print("\nTop 30 policy bigrams")
for phrase, count in ngram_counts(df['Processed_Content'], n=2, top_k=30):
    print(f"{phrase}: {count}")

print("\nTop 30 policy trigrams")
for phrase, count in ngram_counts(df['Processed_Content'], n=3, top_k=30):
    print(f"{phrase}: {count}")
